In [1]:
import tensorflow as tf
import matplotlib.pyplot as plt

In [2]:
train = tf.keras.utils.image_dataset_from_directory(
    'cats_vs_dogs',
    image_size=(224, 224),
    batch_size=32,
    validation_split=0.2,
    subset='training',
    seed=123
)

val = tf.keras.utils.image_dataset_from_directory(
    'cats_vs_dogs',
    image_size=(224, 224),
    batch_size=32,
    validation_split=0.2,
    subset='validation',
    seed=123
)

Found 1000 files belonging to 2 classes.
Using 800 files for training.
Found 1000 files belonging to 2 classes.
Using 200 files for validation.


In [3]:
base_model = tf.keras.applications.ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

In [4]:
base_model.trainable = False

In [5]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(224, 224, 3)),
    tf.keras.layers.Lambda(
        tf.keras.applications.resnet50.preprocess_input
    ),
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(2, activation='softmax')
])

In [6]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [7]:
h1 = model.fit(train, validation_data=val, epochs=3)

Epoch 1/3
25/25 ━━━━━━━━━━━━━━━━━━━━ 118s 4s/step - accuracy: 0.8637 - loss: 0.2750 - val_accuracy: 0.9800 - val_loss: 0.0646
Epoch 2/3
25/25 ━━━━━━━━━━━━━━━━━━━━ 136s 4s/step - accuracy: 0.9900 - loss: 0.0493 - val_accuracy: 0.9650 - val_loss: 0.0801
Epoch 3/3
25/25 ━━━━━━━━━━━━━━━━━━━━ 142s 6s/step - accuracy: 0.9850 - loss: 0.0424 - val_accuracy: 0.9800 - val_loss: 0.0509


In [8]:
base_model.trainable = True

for layer in base_model.layers[:-10]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(0.0001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [9]:
h2 = model.fit(train, validation_data=val, epochs=3)

Epoch 1/3


MemoryError: Exception encountered when calling BatchNormalization.call().

[1m[0m

Arguments received by BatchNormalization.call():
  • inputs=tf.Tensor(shape=(None, 7, 7, 512), dtype=float32)
  • training=True
  • mask=None

In [ ]:
print("Before Fine Tuning:", h1.history['val_accuracy'][-1])
print("After Fine Tuning:", h2.history['val_accuracy'][-1])

In [ ]:
plt.plot(h1.history['val_accuracy'], label='Frozen')
plt.plot(h2.history['val_accuracy'], label='Fine Tuned')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('ResNet50 Transfer Learning')
plt.legend()
plt.show()

In [ ]:
plt.plot(h1.history['val_accuracy'], label='Frozen')
plt.plot(h2.history['val_accuracy'], label='Fine Tuned')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('ResNet50 Transfer Learning')
plt.legend()
plt.show()

In [ ]:
plt.plot(h1.history['val_loss'], label='Frozen')
plt.plot(h2.history['val_loss'], label='Fine Tuned')
plt.xlabel('Epochs')
plt.ylabel('Validation Loss')
plt.title('ResNet50 Validation Loss')
plt.legend()
plt.show()